In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding, Input, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from textblob import TextBlob
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# 加载假新闻和真新闻数据
fake_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--Fake.xlsx', engine='openpyxl')
true_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/fake-and-real-news-dataset--True.xlsx', engine='openpyxl')

# 添加标签
fake_df['label'] = 0
true_df['label'] = 1

# 合并数据
data_df = pd.concat([fake_df, true_df], ignore_index=True)
# 清理数据：删除或填充NaN值，确保所有文本数据都是字符串
data_df['text'].fillna('', inplace=True)
data_df['text'] = data_df['text'].astype(str)
# 提取文本和标签
X = data_df['text'].values
y = data_df['label'].values

# 分割数据集为训练和测试集
X_train_texts, X_test_texts, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 预处理文本数据
max_words = 10000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_texts)
X_train_sequences = tokenizer.texts_to_sequences(X_train_texts)
X_test_sequences = tokenizer.texts_to_sequences(X_test_texts)

X_train = pad_sequences(X_train_sequences, maxlen=max_len)
X_test = pad_sequences(X_test_sequences, maxlen=max_len)

# 将标签转换为浮点数
y_train = y_train.astype(float)
y_test = y_test.astype(float)

# 使用 TextBlob 获取情感分数
def get_sentiment_scores(texts):
    scores = []
    for text in texts:
        blob = TextBlob(text)
        scores.append([blob.sentiment.polarity])
    return np.array(scores)

# 获取情感分数
X_train_sentiments = get_sentiment_scores(X_train_texts)
X_test_sentiments = get_sentiment_scores(X_test_texts)

# 构造基础模型
def create_embedding_model():
    text_input = Input(shape=(150,), dtype=tf.int32, name='text_input')
    sentiment_input = Input(shape=(1,), dtype=tf.float32, name='sentiment_input')
    
    text_embedding = Embedding(input_dim=10000, output_dim=128, input_length=150, name='embedding')(text_input)
    text_flatten = Flatten(name='flatten')(text_embedding)
    
    concatenated = Concatenate()([text_flatten, sentiment_input])
    
    x = Dense(64, activation='relu', name='dense_1')(concatenated)
    output = Dense(1, activation='sigmoid', name='dense_2')(x)
    
    model = Model(inputs=[text_input, sentiment_input], outputs=output)
    return model

# 创建中间模型
def create_middle_model(base_model):
    text_input = base_model.get_layer(name='text_input').input
    embedding_output = base_model.get_layer(name='embedding').output
    middle_model = Model(inputs=text_input, outputs=embedding_output)
    return middle_model

# 使用输入张量计算梯度生成对抗样本
def generate_adversarial_samples(model, middle_model, X_sample, y_sample, sentiment_scores, epsilon=0.1):
    X_sample_tensor = tf.convert_to_tensor(X_sample, dtype=tf.int32)
    y_sample_tensor = tf.convert_to_tensor(y_sample, dtype=tf.float32)
    sentiment_tensor = tf.convert_to_tensor(sentiment_scores, dtype=tf.float32)
    # 用于记录模型前向传播时的操作，以便之后进行反向传播计算梯度。
    with tf.GradientTape() as tape:
        embed_output = middle_model(X_sample_tensor)
        tape.watch(embed_output)
        flatten_output = tf.reshape(embed_output, [-1, 150 * 128])
        concatenated_output = tf.concat([flatten_output, sentiment_tensor], axis=-1)
        predictions = model.layers[-1](model.layers[-2](concatenated_output))
        loss = BinaryCrossentropy()(y_sample_tensor, predictions)

    gradients = tape.gradient(loss, embed_output) #计算嵌入层输出相对于损失的梯度。

    adversarial_samples = tf.argmax(embed_output, axis=-1).numpy()
    return adversarial_samples

# 计算额外的评估指标
def evaluate_model(model, X, sentiments, y_true):
    y_pred = (model.predict([X, sentiments]) > 0.5).astype(int)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return precision, recall, f1

# 聚合对抗性训练数据进行训练
def train_and_evaluate_adv(model, middle_model, X_train, y_train, X_test, y_test, X_train_sentiments, X_test_sentiments, epochs=10, epsilon=0.1):
    history_logs = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    precision_logs = {'train': [], 'val': []}
    recall_logs = {'train': [], 'val': []}
    f1_logs = {'train': [], 'val': []}

    model.compile(optimizer=Adam(learning_rate=1e-4), loss=BinaryCrossentropy(), metrics=['accuracy'])
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")

        # 生成对抗样本
        X_adv_samples = generate_adversarial_samples(model, middle_model, X_train, y_train, X_train_sentiments, epsilon)

        # 合并对抗数据
        X_combined = np.concatenate((X_train, X_adv_samples), axis=0)
        y_combined = np.concatenate((y_train, y_train), axis=0)
        sentiment_combined = np.concatenate((X_train_sentiments, X_train_sentiments), axis=0)

        history = model.fit([X_combined, sentiment_combined], y_combined, epochs=1, batch_size=16, validation_split=0.2, verbose=1)

        for key in history_logs:
            history_logs[key].append(history.history[key][0])

        # 计算训练和验证集上的精确率、召回率和F1分数
        val_split_idx = len(X_combined) * 80 // 100
        precision_train, recall_train, f1_train = evaluate_model(model, X_combined[:val_split_idx], sentiment_combined[:val_split_idx], y_combined[:val_split_idx])
        precision_val, recall_val, f1_val = evaluate_model(model, X_combined[val_split_idx:], sentiment_combined[val_split_idx:], y_combined[val_split_idx:])

        precision_logs['train'].append(precision_train)
        recall_logs['train'].append(recall_train)
        f1_logs['train'].append(f1_train)
        precision_logs['val'].append(precision_val)
        recall_logs['val'].append(recall_val)
        f1_logs['val'].append(f1_val)

    # 在原始测试集上进行评估（不结合对抗样本）
    _, accuracy = model.evaluate([X_test, X_test_sentiments], y_test, verbose=0)
    print(f'对抗训练模型在原始测试集上的准确率: {accuracy:.4f}')

    precision, recall, f1 = evaluate_model(model, X_test, X_test_sentiments, y_test)
    print(f'对抗训练模型在原始测试集上的精确率: {precision:.4f}, 召回率: {recall:.4f}, F1分数: {f1:.4f}')

    # 生成并评估对抗测试数据
    X_test_adv_samples = generate_adversarial_samples(model, middle_model, X_test, y_test, X_test_sentiments, epsilon)

    _, accuracy_adv = model.evaluate([X_test_adv_samples, X_test_sentiments], y_test, verbose=0)
    print(f'对抗训练模型在对抗测试集上的准确率: {accuracy_adv:.4f}')

    precision_adv, recall_adv, f1_adv = evaluate_model(model, X_test_adv_samples, X_test_sentiments, y_test)
    print(f'对抗训练模型在对抗测试集上的精确率: {precision_adv:.4f}, 召回率: {recall_adv:.4f}, F1分数: {f1_adv:.4f}')

    return model, history_logs, precision_logs, recall_logs, f1_logs

# 调用对抗训练过程
model_adv = create_embedding_model()
middle_model_adv = create_middle_model(model_adv)

print("\n训练对抗训练模型...")
model_adv, adv_history, precision_logs, recall_logs, f1_logs = train_and_evaluate_adv(model_adv, middle_model_adv, X_train, y_train, X_test, y_test, X_train_sentiments, X_test_sentiments, epochs=10)


d:\conda\tensorflow-gpu2.2.0\lib\site-packages\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)



训练对抗训练模型...
Epoch 1/10
994/994 [==============================] - 34s 34ms/step - loss: 0.4218 - accuracy: 0.8271 - val_loss: 0.1299 - val_accuracy: 0.9831
Epoch 2/10
994/994 [==============================] - 34s 34ms/step - loss: 0.3047 - accuracy: 0.8706 - val_loss: 0.2062 - val_accuracy: 0.9243
Epoch 3/10
994/994 [==============================] - 34s 34ms/step - loss: 0.1325 - accuracy: 0.9623 - val_loss: 0.1003 - val_accuracy: 0.9635
Epoch 4/10
994/994 [==============================] - 34s 34ms/step - loss: 0.1279 - accuracy: 0.9560 - val_loss: 0.2432 - val_accuracy: 0.9036
Epoch 5/10
994/994 [==============================] - 34s 34ms/step - loss: 0.0747 - accuracy: 0.9776 - val_loss: 0.2212 - val_accuracy: 0.9165
Epoch 6/10
994/994 [==============================] - 34s 34ms/step - loss: 0.0522 - accuracy: 0.9832 - val_loss: 0.2484 - val_accuracy: 0.9064
Epoch 7/10
994/994 [==============================] - 34s 35ms/step - loss: 0.0275 - accuracy: 0.9936 - val_loss: 0.2505 - 